# Fink/LSST — Ajout de `r:visit` et `r:detector` aux light curves

Ce notebook ré-enrichit les fichiers Parquet existants sauvegardés par  
`01_fink_block_lightcurves.ipynb` en ajoutant les colonnes **`r:visit`** et **`r:detector`**,  
indispensables pour sélectionner les visites et exécuter le **DRP Rubin à l'USDF**.

## Stratégie

1. Charger tous les Parquets `*_src.parquet` depuis `data_FINK_BLOCK_LC_01/`
2. Collecter l'ensemble unique de `diaObjectId` présents
3. Re-fetcher via `api/v1/sources` **uniquement les colonnes manquantes** :  
   `r:diaObjectId, r:diaSourceId, r:visit, r:detector, r:midpointMjdTai, r:band, r:x, r:y`
4. Joindre via `r:diaSourceId` (clé unique par détection)
5. Sauvegarder les Parquets enrichis dans `data_FINK_BLOCK_LC_01/` (suffix `_src_v2.parquet`)
6. Bonus : générer `visit_index.csv` — index complet `(visitId, detector, diaObjectId, group, band, mjd)`
   prêt à l'emploi pour le Butler Rubin

## Colonnes ajoutées

| Colonne | Description | Usage DRP |
|---------|-------------|----------|
| `r:visit` | Numéro de visite (int 13 chiffres, ex: `2026030900027`) | `butler.get('visitId')` |
| `r:detector` | Numéro du CCD (0–188) | `butler.get('detector')` |
| `r:x` | Position x en pixels sur le CCD | Localisation dans le patch |
| `r:y` | Position y en pixels sur le CCD | Localisation dans le patch |

**Format `r:visit`** : `AAAAMMJJNNNNN` — ex. `2026030900027` = 9 mars 2026, visite #27.  
Utilisable directement comme `visitId` dans le Butler pour lancer `isr`, `calibrate`, etc.

> **Note** : les Parquets `*_fp.parquet` (forced photometry) ne contiennent pas
> `r:diaSourceId` donc on ne peut pas joindre proprement — mais `r:visit` n'est
> pas défini pour la forced phot (qui couvre toutes les visites de toute façon).
> L'index de visite est donc construit sur les `_src` uniquement.


## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
import glob
import time
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # répertoire des parquets existants

# Colonnes à re-fetcher depuis l'API (légères — seulement ce qui manque)
# r:diaSourceId est la clé de jointure avec les parquets existants
COLS_VISIT = "r:diaObjectId,r:diaSourceId,r:visit,r:detector,r:midpointMjdTai,r:band,r:x,r:y"

# Délai entre les appels API (secondes) — pour ne pas surcharger le broker
API_SLEEP = 0.25

print(f"Data directory : {os.path.abspath(DIR_DATA)}")
print(f"Colonnes fetched : {COLS_VISIT}")

## 2. API wrapper

In [ ]:
def fetch_sources_visit(diaObjectId: str, columns: str = COLS_VISIT, timeout: int = 60) -> pd.DataFrame:
    """
    Récupère depuis Fink uniquement les colonnes liées aux visites
    pour un diaObjectId donné.
    Retourne un DataFrame avec r:diaSourceId comme clé de jointure.
    """
    try:
        r = requests.post(
            f"{FINK_API}/api/v1/sources",
            json={"diaObjectId": str(diaObjectId), "columns": columns, "output-format": "json"},
            timeout=timeout,
        )
        r.raise_for_status()
        data = r.json()
        if not data:
            return pd.DataFrame()
        df = pd.DataFrame(data)
        # Convertir r:visit en entier (parfois retourné comme float)
        if "r:visit" in df.columns:
            df["r:visit"] = pd.to_numeric(df["r:visit"], errors="coerce").astype("Int64")
        if "r:detector" in df.columns:
            df["r:detector"] = pd.to_numeric(df["r:detector"], errors="coerce").astype("Int64")
        return df
    except Exception as e:
        print(f"    ERROR fetch {diaObjectId}: {e}")
        return pd.DataFrame()


print("API wrapper défini.")

## 3. Charger les parquets `_src` existants et extraire les diaObjectId

In [ ]:
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))
print(f"{len(src_files)} files _src.parquet trouvés :")
for f in src_files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Charger tous les _src en mémoire et collecter les diaObjectId uniques
src_cache = {}  # group -> DataFrame
all_oids = set()

for path in src_files:
    group = os.path.basename(path).replace("_src.parquet", "")
    df = pd.read_parquet(path)
    src_cache[group] = df
    if "diaObjectId" in df.columns:
        all_oids.update(df["diaObjectId"].astype(str).unique())
    print(f"  {group:40s}  {len(df):6d} lignes    colonnes: {list(df.columns[:6])}...")

all_oids = sorted(all_oids)
print(f"\nTotal diaObjectId uniques : {len(all_oids)}")

# Vérifier si r:visit est déjà présent
sample_group = next(iter(src_cache))
sample_cols = list(src_cache[sample_group].columns)
print(f'\nColonnes actuelles (groupe "{sample_group}") :')
print(sample_cols)
if "r:visit" in sample_cols:
    print("\n✅ r:visit est DÉJÀ présent — ré-enrichissement optionnel.")
else:
    print("\n⚠️  r:visit ABSENT — ré-enrichissement nécessaire.")

In [ ]:
src_cache.keys()

## 6. Générer `visit_index.csv` — index complet pour le Butler Rubin

In [ ]:
# Construire l'index de visite en lisant les _v2 sauvegardés
index_frames = []

v2_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))
print(f"{len(v2_files)} fichiers _v2 trouvés.")

for path in v2_files:
    group = os.path.basename(path).replace("_src_v2.parquet", "")
    df = pd.read_parquet(path)
    if "r:visit" not in df.columns:
        continue
    # Sélectionner les colonnes utiles pour l'index
    keep = [
        "diaObjectId",
        "r:diaSourceId",
        "r:visit",
        "r:detector",
        "r:band",
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:x",
        "r:y",
    ]
    keep = [c for c in keep if c in df.columns]
    tmp = df[keep].copy()
    tmp["group"] = group
    index_frames.append(tmp)

if index_frames:
    df_index = (
        pd.concat(index_frames, ignore_index=True)
        .dropna(subset=["r:visit"])
        .sort_values(["r:visit", "r:detector", "r:band"])
        .reset_index(drop=True)
    )
    # Convertir r:visit en int (enlever les .0)
    df_index["r:visit"] = df_index["r:visit"].astype("Int64")
    df_index["r:detector"] = df_index["r:detector"].astype("Int64")

    out_path = os.path.join(DIR_DATA, "visit_index.csv")
    df_index.to_csv(out_path, index=False)

    print(f"\nIndex sauvegardé : {out_path}")
    print(f"  Lignes               : {len(df_index)}")
    print(f"  Visites uniques      : {df_index['r:visit'].nunique()}")
    print(f"  Objets uniques       : {df_index['diaObjectId'].nunique()}")
    print(f"  Groupes              : {df_index['group'].unique().tolist()}")
    print(f"\nExtrait (10 lignes) :")
    print(
        df_index[["r:visit", "r:detector", "r:band", "diaObjectId", "group", "r:midpointMjdTai"]]
        .head(10)
        .to_string()
    )
else:
    print("Aucun fichier v2 disponible pour construire l'index.")

## 7. Extraction des visitId pour le DRP Rubin

Exemples d'utilisation de l'index pour sélectionner des visites à traiter.

In [ ]:
# ── Exemple 1 : toutes les visites d'un groupe calibrateur ────────────────────
TARGET_GROUP = "gaia_star_stable_hq"  # changer selon le groupe souhaité

if not df_index.empty and TARGET_GROUP in df_index["group"].values:
    df_group = df_index[df_index["group"] == TARGET_GROUP]
    visits_group = sorted(df_group["r:visit"].dropna().unique())
    print(f'Groupe "{TARGET_GROUP}" — {len(visits_group)} visites uniques :')
    print(visits_group[:20], "..." if len(visits_group) > 20 else "")
else:
    print(f'Groupe "{TARGET_GROUP}" non trouvé. Groupes disponibles :')
    if not df_index.empty:
        print(df_index["group"].unique().tolist())

In [ ]:
# ── Exemple 2 : toutes les visites, tous groupes confondus ────────────────────
if not df_index.empty:
    all_visits = sorted(df_index["r:visit"].dropna().unique())
    print(f"Toutes visites (tous groupes) : {len(all_visits)} uniques")
    print("Exemples :", all_visits[:10])

    # Distribution par bande
    print("\nVisites par bande :")
    print(df_index.groupby("r:band")["r:visit"].nunique().rename("n_visits_uniques"))

    # Distribution par groupe
    print("\nVisites par groupe :")
    print(
        df_index.groupby("group")["r:visit"].nunique().sort_values(ascending=False).rename("n_visits_uniques")
    )

In [ ]:
# ── Exemple 3 : extraire la liste (visitId, detector) pour le Butler ──────────
# Format attendu par le DRP Rubin : liste de (visitId, detector)

if not df_index.empty:
    # Pour les étoiles Gaia stables uniquement, bande r
    mask = df_index["group"].str.startswith("gaia_star_stable") & (df_index["r:band"] == "r")
    df_drp = (
        df_index[mask][["r:visit", "r:detector"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["r:visit", "r:detector"])
        .reset_index(drop=True)
    )
    print(f"Paires (visitId, detector) pour étoiles Gaia stables en bande r :")
    print(f"  {len(df_drp)} paires uniques")
    print(df_drp.head(20).to_string())

    # Sauvegarder pour usage direct
    drp_path = os.path.join(DIR_DATA, "drp_visit_detector_gaia_r.csv")
    df_drp.to_csv(drp_path, index=False)
    print(f"\nSauvegardé : {drp_path}")

In [ ]:
# ── Exemple 4 : snippet Butler Rubin prêt à l'emploi (USDF) ──────────────────

butler_snippet = """
# ── Snippet Butler Rubin (USDF) ───────────────────────────────────────────────
# À adapter selon votre collection et pipeline

import pandas as pd
from lsst.daf.butler import Butler

# Charger l'index
df_index = pd.read_csv("data_FINK_BLOCK_LC_01/visit_index.csv")

# Sélectionner les visites d'intérêt (ex: étoiles Gaia stables, bande r)
mask = (
    df_index["group"].str.startswith("gaia_star_stable") &
    (df_index["r:band"] == "r")
)
visits_r = sorted(df_index[mask]["r:visit"].dropna().unique().astype(int))

# Initialiser le Butler
repo   = "/repo/embargo"          # adapter à votre collection USDF
coll   = "LSSTComCam/runs/DRP/..."
butler = Butler(repo, collections=[coll])

# Récupérer les calexps pour ces visites
for visit_id in visits_r[:5]:    # tester sur 5 d'abord
    try:
        calexp = butler.get("calexp", visit=int(visit_id), detector=0)
        print(f"visit {visit_id} OK  shape: {calexp.image.array.shape}")
    except Exception as e:
        print(f"visit {visit_id} ERREUR: {e}")
# ──────────────────────────────────────────────────────────────────────────────
"""
print(butler_snippet)

## 8. Vérification du contenu des parquets enrichis

In [ ]:
# Charger un fichier _v2 de contrôle et afficher ses colonnes
v2_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))
if v2_files:
    sample_path = v2_files[0]
    df_check = pd.read_parquet(sample_path)
    print(f"Vérification : {os.path.basename(sample_path)}")
    print(f"Shape : {df_check.shape}")
    print(f"\nColonnes (avec taux de remplissage) :")
    for col in df_check.columns:
        non_null = df_check[col].notna().sum()
        print(f"  {col:50s}  non-null: {non_null}/{len(df_check)}")
    print(f"\nExemple (5 lignes, colonnes clés) :")
    show_cols = [
        c
        for c in [
            "diaObjectId",
            "r:diaSourceId",
            "r:visit",
            "r:detector",
            "r:band",
            "r:midpointMjdTai",
            "r:psfFlux",
            "r:x",
            "r:y",
        ]
        if c in df_check.columns
    ]
    print(df_check[show_cols].dropna(subset=["r:visit"]).head(5).to_string())
else:
    print("Aucun fichier _v2 trouvé.")